<a href="https://colab.research.google.com/github/cras-lab/ML-examples/blob/main/Kmeans_LLM_SemanticClustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 임포트 한다.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances
import pandas as pd

예제 문장을 설정한다.<BR>
좋은점과 아쉬운 점 등이 섞여 있도록 문장을 구성

In [ ]:
sentences = [
    "강의 속도가 너무 빨라서 따라가기 어려웠습니다.",
    "조금 천천히 설명해 주시면 좋겠습니다.",
    "설명이 전반적으로 명확하고 이해하기 쉬웠습니다.",
    "예제가 많아서 개념을 이해하는 데 큰 도움이 됐습니다.",
    "실습 예제가 더 많았으면 좋겠습니다.",
    "과제가 생각보다 어려웠지만 많이 배웠습니다.",
    "과제 난이도가 너무 높아서 부담이 컸습니다.",
    "실습 시간이 부족해서 아쉬웠습니다.",
    "코드 시연이 있어서 좋았습니다.",
    "이론 설명이 체계적이라 좋았습니다.",
    "시험 범위를 조금 더 구체적으로 알려주시면 좋겠습니다.",
    "평가 기준이 명확했으면 좋겠습니다.",
    "실제 사례를 많이 보여줘서 흥미로웠습니다.",
    "수업 자료가 잘 정리되어 있어서 복습하기 좋았습니다.",
    "과제 피드백을 더 자세히 받고 싶습니다."
]

문장을 임베딩한다.<BR>
Hugging Face에 공개된  MiniLM Transformer를 이용하며,<BR>필요시 OpenAI나 Gemini 등의 embedding으로 바꿔도 된다.

In [ ]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(sentences)

임베딩이 어떻게 되었는지  차원과 내용을 출력해 본다.

In [ ]:
print( f"모델의 임베딩 차원: ", model.get_embedding_dimension())
print(embeddings)

클러스터 개수를 4로 설정한다. Default는 8

In [16]:
k = 4

실제로 Kmeans 객체를 생성한다.

In [17]:
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

embeddings를 대상으로 클러스터링 실행

In [24]:
labels = kmeans.fit_predict(embeddings)

레이블 결과를 sentence와 함께 딕셔너리에 저장

In [ ]:
df = pd.DataFrame({
    "sentence": sentences,
    "cluster": labels
})

전체군집결과를 출력해 본다.

In [ ]:
print("=== 전체 군집 결과 ===")
print(df.sort_values("cluster").to_string(index=False))

군집별 대표 문장을 출력해 본다. <BR>
centroid와 가장 가까운 문장을 출력해 본다.

In [ ]:
print("=== 군집별 대표 문장 ===")
for cluster_id in range(k):
    cluster_sentences = df[df["cluster"] == cluster_id]["sentence"].tolist()
    cluster_embeddings = embeddings[df[df["cluster"] == cluster_id].index]

    centroid = kmeans.cluster_centers_[cluster_id].reshape(1, -1)
    distances = cosine_distances(cluster_embeddings, centroid).reshape(-1)
    representative_idx = distances.argmin()
    representative_sentence = cluster_sentences[representative_idx]

    print(f"\n[Cluster {cluster_id}]")
    print("대표 문장:", representative_sentence)
    print("포함 문장:")
    for s in cluster_sentences:
        print("-", s)